# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring the FAIR² dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata using mlcroissant
dataset = mlc.Dataset(croissant_url)

# Access metadata attributes
meta = dataset.metadata
print(f"{meta.name}: {meta.description}\n")

## 2. Data Overview
Review available record sets and entity IDs. Each field/column and record set is referenced by its `@id`.

In [ ]:
# List available Record Sets using their @id
print("Available RecordSets (referenced by @id):")
for rs in dataset.metadata.record_sets:
    print(f"- RecordSet name: {rs.name}, @id: {rs.id}")
    # For each RecordSet, list its fields and their @id
    print("  Fields:")
    for field in rs.fields:
        print(f"    - Field name: {field.name}, @id: {field.id}, dataType: {field.data_type}")
    print()

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.

Below, we extract data for all primary record sets using their `@id`. We use a dictionary to store the DataFrames.

In [ ]:
# Collect all RecordSet @id values
record_set_ids = [rs.id for rs in dataset.metadata.record_sets]
print("RecordSet @ids to extract:")
print(record_set_ids)

# Dictionary of DataFrames
dataframes = {}
for rs_id in record_set_ids:
    records = list(dataset.records(record_set=rs_id))
    if records:
        dataframes[rs_id] = pd.DataFrame(records)
        print(f"Loaded data for {rs_id} into DataFrame with shape {dataframes[rs_id].shape}")
        print(f"Columns: {dataframes[rs_id].columns.tolist()}")
        print(dataframes[rs_id].head(), "\n")
    else:
        print(f"No rows found for record_set {rs_id}\n")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. Operations include removing outliers, transforming distributions, or grouping by key attributes to prepare for further analysis.

**Select a RecordSet and relevant fields using their `@id` as shown in the overview.**

*Example below selects the first RecordSet and the first numeric field found in its schema for demonstration purposes.*

In [ ]:
import numpy as np

# Choose the first RecordSet as example
if not record_set_ids:
    print('No record sets found!')
else:
    main_rs_id = record_set_ids[0]
    df = dataframes.get(main_rs_id)
    # Try to find a numeric field
    numeric_field_id = None
    group_field_id = None
    for field in dataset.metadata.get_record_set(main_rs_id).fields:
        if field.data_type in ('Float', 'Integer', 'Number') and not numeric_field_id:
            numeric_field_id = field.id
        if not group_field_id and field.data_type == 'Text':
            group_field_id = field.id
    print(f"Selected numeric field: {numeric_field_id}")
    print(f"Selected group field: {group_field_id}")

    if numeric_field_id and numeric_field_id in df.columns:
        # Drop rows with missing values in numeric field
        filtered_df = df.dropna(subset=[numeric_field_id])
        # For demonstration, use the median as threshold if values substantial
        try:
            threshold = np.median(filtered_df[numeric_field_id])
            filtered_df = filtered_df[filtered_df[numeric_field_id].astype(float) > threshold]
            print(f"Filtered records with {numeric_field_id} > {threshold}:")
            print(filtered_df.head())

            filtered_df[f"{numeric_field_id}_normalized"] = (
                (filtered_df[numeric_field_id].astype(float) - filtered_df[numeric_field_id].astype(float).mean()) /
                filtered_df[numeric_field_id].astype(float).std()
            )
            print(f"Normalized {numeric_field_id} for filtered records:")
            print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

            if group_field_id and group_field_id in filtered_df.columns:
                grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
                print(f"Grouped data by {group_field_id} (mean of {numeric_field_id}):")
                print(grouped_df.head())
        except Exception as e:
            print(f"Unable to perform numeric analysis: {e}")
    else:
        print("No numeric field available in the selected RecordSet.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

*The following cell generates a histogram and a boxplot for the selected numeric field, and a bar plot for the group field means if available.*

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if not record_set_ids or main_rs_id not in dataframes:
    print('No data available for visualization.')
else:
    df = dataframes[main_rs_id]
    if numeric_field_id and numeric_field_id in df.columns:
        plt.figure(figsize=(12,5))
        sns.histplot(df[numeric_field_id].dropna().astype(float), bins=30, kde=True)
        plt.title(f'Distribution of {numeric_field_id}')
        plt.xlabel(numeric_field_id)
        plt.show()

        plt.figure(figsize=(8,4))
        sns.boxplot(x=df[numeric_field_id].dropna().astype(float))
        plt.title(f'Boxplot of {numeric_field_id}')
        plt.show()

        if group_field_id and group_field_id in df.columns:
            plt.figure(figsize=(12,6))
            means = df.groupby(group_field_id)[numeric_field_id].mean().sort_values(ascending=False)
            means = means.dropna().head(10) # top 10 only for clarity
            means.plot(kind='bar')
            plt.ylabel(f'Mean {numeric_field_id}')
            plt.title(f'Mean {numeric_field_id} by {group_field_id} (top 10)')
            plt.show()
    else:
        print('No numeric field available to plot.')

## 6. Conclusion
In this notebook, we've demonstrated how to use the `mlcroissant` library to load, inspect, analyze, and visualize data from a FAIR² dataset, referencing all entities (record sets, fields) by their `@id`. You can adapt this workflow for further data processing or machine learning tasks depending on your research needs.